In [1]:
import os, glob
from collections import Counter

root = "/kaggle/input"
print(os.listdir(root))
print("CSV files:", glob.glob(f"{root}/**/*.csv", recursive=True))

exts = Counter(os.path.splitext(f)[1].lower()
               for _, _, fs in os.walk(root) for f in fs)
print(exts)

['datasets']
CSV files: ['/kaggle/input/datasets/mahdavi1202/skin-cancer/metadata.csv']
Counter({'.png': 2298, '.csv': 1})


In [2]:
import os, glob
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

root = "/kaggle/input"
meta_path = glob.glob(f"{root}/**/metadata.csv", recursive=True)[0]
df = pd.read_csv(meta_path)
print(df.columns.tolist())
print(df.head(3))

paths = {os.path.basename(p): p for p in glob.glob(f"{root}/**/*.png", recursive=True)}
df["path"] = df["img_id"].map(paths)
if df["path"].isna().all():                      # img_id might lack the extension
    df["path"] = (df["img_id"].astype(str) + ".png").map(paths)
print("missing files:", df["path"].isna().sum())
df = df.dropna(subset=["path"]).reset_index(drop=True)

print(df["diagnostic"].value_counts())
print("patients:", df["patient_id"].nunique(), "| images:", len(df))

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
folds = list(sgkf.split(df, df["diagnostic"], groups=df["patient_id"]))
df["split"] = "train"
df.loc[folds[0][1], "split"] = "val"
df.loc[folds[1][1], "split"] = "test"
print(pd.crosstab(df["diagnostic"], df["split"]))
df.to_csv("/kaggle/working/split.csv", index=False)

['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation', 'img_id', 'biopsed']
  patient_id  lesion_id  smoke  drink background_father background_mother  \
0   PAT_1516       1765    NaN    NaN               NaN               NaN   
1     PAT_46        881  False  False         POMERANIA         POMERANIA   
2   PAT_1545       1867    NaN    NaN               NaN               NaN   

   age pesticide  gender skin_cancer_history  ... diameter_2 diagnostic  \
0    8       NaN     NaN                 NaN  ...        NaN        NEV   
1   55     False  FEMALE                True  ...        5.0        BCC   
2   77       NaN     NaN                 NaN  ...        NaN        ACK   

    itch   grew   hurt  changed  bleed elevati

In [3]:
import torch, torch.nn as nn, timm, numpy as np, pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import StratifiedGroupKFold

dev = "cuda"
df = pd.read_csv("/kaggle/working/split.csv")
classes = sorted(df.diagnostic.unique()); c2i = {c: i for i, c in enumerate(classes)}
df["y"] = df.diagnostic.map(c2i)
print(classes)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df["fold"] = -1
for k, (_, te) in enumerate(sgkf.split(df, df.y, groups=df.patient_id)):
    df.loc[te, "fold"] = k
print(pd.crosstab(df.diagnostic, df.fold))   # check MEL appears in every fold

# resize once and keep in RAM (the raw PNGs are big, so reloading each epoch is slow)
imgs = [Image.open(p).convert("RGB").resize((256, 256)) for p in df.path]
print("cached", len(imgs))

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']
fold          0    1    2    3    4
diagnostic                         
ACK         144  135  127  170  154
BCC         173  171  175  134  192
MEL          13    9    5   14   11
NEV          45   48   54   51   46
SCC          37   35   38   42   40
SEK          48   43   49   48   47
cached 2298


In [4]:
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomVerticalFlip(), T.RandomRotation(20),
                      T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), norm])
tf_eval = T.Compose([T.CenterCrop(224), T.ToTensor(), norm])

class DS(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = list(idx), tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; return self.tf(imgs[j]), int(df.y.iloc[j])

def run_fold(k, epochs=12, bs=32):
    tr_idx = df.index[df.fold != k]; te_idx = df.index[df.fold == k]
    tl = DataLoader(DS(tr_idx, tf_train), bs, shuffle=True, num_workers=2, drop_last=True)
    el = DataLoader(DS(te_idx, tf_eval), 64, num_workers=2)
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(classes)).to(dev)
    cnt = df.loc[tr_idx, "y"].value_counts().sort_index().values
    w = torch.tensor(cnt.sum() / (len(cnt) * cnt), dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w)
    scaler = torch.cuda.amp.GradScaler()
    for ep in range(epochs):
        if ep == 0:      # head only for the first 3 epochs
            for n, p in model.named_parameters(): p.requires_grad = "classifier" in n
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        if ep == 3:      # then fine-tune everything with a lower LR
            for p in model.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs - 3)
        model.train(); tot = 0
        for x, y in tl:
            x, y = x.to(dev), y.to(dev); opt.zero_grad()
            with torch.autocast("cuda"): loss = lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        if ep >= 3: sched.step()
        print(f"fold {k} ep {ep} loss {tot/len(tl):.3f}")
    model.eval(); P = []
    with torch.no_grad():
        for x, _ in el: P.append(torch.softmax(model(x.to(dev)), 1).cpu().numpy())
    return te_idx, np.concatenate(P), model

# debug on ONE fold first
te_idx, probs, _ = run_fold(0)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/1971251004.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


fold 0 ep 0 loss 3.894
fold 0 ep 1 loss 3.146
fold 0 ep 2 loss 2.748
fold 0 ep 3 loss 2.198
fold 0 ep 4 loss 1.529
fold 0 ep 5 loss 1.158
fold 0 ep 6 loss 0.849
fold 0 ep 7 loss 0.809
fold 0 ep 8 loss 0.700
fold 0 ep 9 loss 0.659
fold 0 ep 10 loss 0.589
fold 0 ep 11 loss 0.622


In [5]:
oof = np.zeros((len(df), len(classes)))
oof[te_idx] = probs        # fold 0, from your earlier run

for k in range(1, 5):
    te_idx_k, probs_k, model_k = run_fold(k)
    oof[te_idx_k] = probs_k
    torch.save(model_k.state_dict(), f"/kaggle/working/effb0_fold{k}.pt")
    np.save("/kaggle/working/oof_probs.npy", oof)
    print(f"=== fold {k} done ===")
    del model_k; torch.cuda.empty_cache()

print((oof.sum(1) == 0).sum(), "rows without predictions (should be 0)")

/tmp/ipykernel_23/1971251004.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


fold 1 ep 0 loss 3.477
fold 1 ep 1 loss 3.098
fold 1 ep 2 loss 2.793
fold 1 ep 3 loss 2.194
fold 1 ep 4 loss 1.479
fold 1 ep 5 loss 1.097
fold 1 ep 6 loss 0.923
fold 1 ep 7 loss 0.725
fold 1 ep 8 loss 0.687
fold 1 ep 9 loss 0.617
fold 1 ep 10 loss 0.582
fold 1 ep 11 loss 0.538
=== fold 1 done ===
fold 2 ep 0 loss 3.676
fold 2 ep 1 loss 2.958
fold 2 ep 2 loss 2.645
fold 2 ep 3 loss 2.303
fold 2 ep 4 loss 1.483
fold 2 ep 5 loss 1.137
fold 2 ep 6 loss 0.959
fold 2 ep 7 loss 0.745
fold 2 ep 8 loss 0.730
fold 2 ep 9 loss 0.597
fold 2 ep 10 loss 0.593
fold 2 ep 11 loss 0.593
=== fold 2 done ===
fold 3 ep 0 loss 3.674
fold 3 ep 1 loss 3.131
fold 3 ep 2 loss 2.713
fold 3 ep 3 loss 2.208
fold 3 ep 4 loss 1.520
fold 3 ep 5 loss 1.141
fold 3 ep 6 loss 0.884
fold 3 ep 7 loss 0.800
fold 3 ep 8 loss 0.679
fold 3 ep 9 loss 0.603
fold 3 ep 10 loss 0.595
fold 3 ep 11 loss 0.535
=== fold 3 done ===
fold 4 ep 0 loss 3.381
fold 4 ep 1 loss 2.903
fold 4 ep 2 loss 2.547
fold 4 ep 3 loss 2.276
fold 4 ep 4 lo

In [6]:
from sklearn.metrics import classification_report
y0 = df.y.values[te_idx]
print(classification_report(y0, probs.argmax(1), target_names=classes, digits=3, zero_division=0))

              precision    recall  f1-score   support

         ACK      0.672     0.556     0.608       144
         BCC      0.727     0.647     0.685       173
         MEL      0.333     0.462     0.387        13
         NEV      0.489     0.489     0.489        45
         SCC      0.227     0.405     0.291        37
         SEK      0.448     0.542     0.491        48

    accuracy                          0.567       460
   macro avg      0.483     0.517     0.492       460
weighted avg      0.606     0.567     0.581       460



In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y = df.y.values
pred = oof.argmax(1)
print(classification_report(y, pred, target_names=classes, digits=3))
print(classes); print(confusion_matrix(y, pred))
for i, c in enumerate(classes):
    print(c, "AUC", round(roc_auc_score(y == i, oof[:, i]), 3))

              precision    recall  f1-score   support

         ACK      0.691     0.566     0.622       730
         BCC      0.664     0.588     0.624       845
         MEL      0.264     0.442     0.331        52
         NEV      0.505     0.602     0.550       244
         SCC      0.190     0.281     0.227       192
         SEK      0.436     0.536     0.481       235

    accuracy                          0.548      2298
   macro avg      0.458     0.503     0.472      2298
weighted avg      0.583     0.548     0.561      2298

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']
[[413 128  14  35  76  64]
 [116 497  12  51 139  30]
 [  1   1  23  14   2  11]
 [  6  28  12 147   4  47]
 [ 39  79   2   7  54  11]
 [ 23  16  24  37   9 126]]
ACK AUC 0.834
BCC AUC 0.809
MEL AUC 0.911
NEV AUC 0.902
SCC AUC 0.691
SEK AUC 0.859


In [8]:
# ---- metadata features (no leaky columns) ----
bool_cols = ["itch","grew","hurt","changed","bleed","elevation","skin_cancer_history","cancer_history"]
M = pd.DataFrame(index=df.index)
for c in bool_cols:
    s = df[c].astype(str).str.upper().map({"TRUE":1.0, "FALSE":0.0})
    M[c] = s.fillna(0.5); M[c+"_na"] = s.isna().astype(float)
for c, scale in [("age",100), ("diameter_1",20), ("diameter_2",20)]:
    s = pd.to_numeric(df[c], errors="coerce")
    M[c] = s.fillna(0)/scale; M[c+"_na"] = s.isna().astype(float)
M = pd.concat([M,
    pd.get_dummies(df["region"].fillna("NA"), prefix="reg").astype(float),
    pd.get_dummies(df["gender"].fillna("NA"), prefix="g").astype(float)], axis=1)
meta = torch.tensor(M.values, dtype=torch.float32)
print("meta features:", meta.shape)

class DSM(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = list(idx), tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; return self.tf(imgs[j]), meta[j], int(df.y.iloc[j])

class Fusion(nn.Module):
    def __init__(self, nmeta, ncls):
        super().__init__()
        self.bb = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
        self.mm = nn.Sequential(nn.Linear(nmeta, 64), nn.ReLU(), nn.Dropout(0.3))
        self.head = nn.Linear(self.bb.num_features + 64, ncls)
    def forward(self, x, m): return self.head(torch.cat([self.bb(x), self.mm(m)], 1))

def run_fold_meta(k, epochs=12, bs=32):
    tr_idx = df.index[df.fold != k]; te_idx = df.index[df.fold == k]
    tl = DataLoader(DSM(tr_idx, tf_train), bs, shuffle=True, num_workers=2, drop_last=True)
    el = DataLoader(DSM(te_idx, tf_eval), 64, num_workers=2)
    model = Fusion(meta.shape[1], len(classes)).to(dev)
    cnt = df.loc[tr_idx, "y"].value_counts().sort_index().values
    w = torch.tensor((cnt.sum()/(len(cnt)*cnt))**0.5, dtype=torch.float32, device=dev)  # softer weights
    lossf = nn.CrossEntropyLoss(weight=w); scaler = torch.amp.GradScaler("cuda")
    for ep in range(epochs):
        if ep == 0:
            for p in model.bb.parameters(): p.requires_grad = False
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        if ep == 3:
            for p in model.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs-3)
        model.train(); tot = 0
        for x, m, y in tl:
            x, m, y = x.to(dev), m.to(dev), y.to(dev); opt.zero_grad()
            with torch.autocast("cuda"): loss = lossf(model(x, m), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        if ep >= 3: sched.step()
        print(f"fold {k} ep {ep} loss {tot/len(tl):.3f}")
    model.eval(); P = []
    with torch.no_grad():
        for x, m, _ in el: P.append(torch.softmax(model(x.to(dev), m.to(dev)), 1).cpu().numpy())
    return te_idx, np.concatenate(P), model

oof_base = oof.copy(); np.save("/kaggle/working/oof_base.npy", oof_base)
oof = np.zeros((len(df), len(classes)))
for k in range(5):
    ti, pk, mk = run_fold_meta(k)
    oof[ti] = pk
    torch.save(mk.state_dict(), f"/kaggle/working/fusion_fold{k}.pt")
    np.save("/kaggle/working/oof_fusion.npy", oof)
    print(f"=== fold {k} done ===")
    del mk; torch.cuda.empty_cache()

meta features: torch.Size([2298, 39])
fold 0 ep 0 loss 1.430
fold 0 ep 1 loss 1.043
fold 0 ep 2 loss 0.874
fold 0 ep 3 loss 0.755
fold 0 ep 4 loss 0.586
fold 0 ep 5 loss 0.494
fold 0 ep 6 loss 0.415
fold 0 ep 7 loss 0.376
fold 0 ep 8 loss 0.325
fold 0 ep 9 loss 0.298
fold 0 ep 10 loss 0.283
fold 0 ep 11 loss 0.280
=== fold 0 done ===
fold 1 ep 0 loss 1.444
fold 1 ep 1 loss 1.020
fold 1 ep 2 loss 0.852
fold 1 ep 3 loss 0.732
fold 1 ep 4 loss 0.571
fold 1 ep 5 loss 0.485
fold 1 ep 6 loss 0.410
fold 1 ep 7 loss 0.368
fold 1 ep 8 loss 0.323
fold 1 ep 9 loss 0.298
fold 1 ep 10 loss 0.281
fold 1 ep 11 loss 0.252
=== fold 1 done ===
fold 2 ep 0 loss 1.467
fold 2 ep 1 loss 1.050
fold 2 ep 2 loss 0.875
fold 2 ep 3 loss 0.749
fold 2 ep 4 loss 0.607
fold 2 ep 5 loss 0.492
fold 2 ep 6 loss 0.446
fold 2 ep 7 loss 0.385
fold 2 ep 8 loss 0.343
fold 2 ep 9 loss 0.313
fold 2 ep 10 loss 0.306
fold 2 ep 11 loss 0.287
=== fold 2 done ===
fold 3 ep 0 loss 1.448
fold 3 ep 1 loss 1.023
fold 3 ep 2 loss 0.852

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
y = df.y.values
print((oof.sum(1) == 0).sum(), "rows without predictions (should be 0)")

def report(o, name):
    pred = o.argmax(1)
    print("=====", name, "=====")
    print(classification_report(y, pred, target_names=classes, digits=3))
    print(confusion_matrix(y, pred))
    print({c: round(roc_auc_score(y == i, o[:, i]), 3) for i, c in enumerate(classes)})

report(oof_base, "baseline (image only)")
report(oof, "fusion (image + metadata)")

# screening view: flag = ACK/BCC/MEL/SCC, benign = NEV/SEK
flag_idx = [classes.index(c) for c in ["ACK", "BCC", "MEL", "SCC"]]
is_flag = np.isin(y, flag_idx)
for name, o in [("baseline", oof_base), ("fusion", oof)]:
    p_flag = o[:, flag_idx].sum(1)
    for t in (0.5, 0.4, 0.3):
        pr = p_flag >= t
        print(name, "thr", t, "sens", round(pr[is_flag].mean(), 3),
              "spec", round((~pr)[~is_flag].mean(), 3))

0 rows without predictions (should be 0)
===== baseline (image only) =====
              precision    recall  f1-score   support

         ACK      0.691     0.566     0.622       730
         BCC      0.664     0.588     0.624       845
         MEL      0.264     0.442     0.331        52
         NEV      0.505     0.602     0.550       244
         SCC      0.190     0.281     0.227       192
         SEK      0.436     0.536     0.481       235

    accuracy                          0.548      2298
   macro avg      0.458     0.503     0.472      2298
weighted avg      0.583     0.548     0.561      2298

[[413 128  14  35  76  64]
 [116 497  12  51 139  30]
 [  1   1  23  14   2  11]
 [  6  28  12 147   4  47]
 [ 39  79   2   7  54  11]
 [ 23  16  24  37   9 126]]
{'ACK': np.float64(0.834), 'BCC': np.float64(0.809), 'MEL': np.float64(0.911), 'NEV': np.float64(0.902), 'SCC': np.float64(0.691), 'SEK': np.float64(0.859)}
===== fusion (image + metadata) =====
              precision 

In [10]:
from sklearn.metrics import f1_score, recall_score

bool_cols = ["itch","grew","hurt","changed","bleed","elevation","skin_cancer_history","cancer_history"]

def build_meta_clean():
    # NO "_na" columns. Missing values are filled with neutral values instead.
    M2 = pd.DataFrame(index=df.index)
    for c in bool_cols:
        M2[c] = df[c].astype(str).str.upper().map({"TRUE": 1.0, "FALSE": 0.0}).fillna(0.0)
    for c, scale in [("age", 100), ("diameter_1", 20), ("diameter_2", 20)]:
        s = pd.to_numeric(df[c], errors="coerce")
        M2[c] = s.fillna(s.median()) / scale
    M2 = pd.concat([M2,
        pd.get_dummies(df["region"].fillna("NA"), prefix="reg").astype(float),
        pd.get_dummies(df["gender"].fillna("NA"), prefix="g").astype(float)], axis=1)
    return torch.tensor(M2.values, dtype=torch.float32)

class DSG(Dataset):
    def __init__(self, idx, tf, imgs_, meta_):
        self.idx, self.tf, self.im, self.m = list(idx), tf, imgs_, meta_
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; return self.tf(self.im[j]), self.m[j], int(df.y.iloc[j])

class FusionG(nn.Module):
    def __init__(self, nmeta, ncls, backbone):
        super().__init__()
        self.bb = timm.create_model(backbone, pretrained=True, num_classes=0)
        self.mm = nn.Sequential(nn.Linear(nmeta, 64), nn.ReLU(), nn.Dropout(0.3))
        self.head = nn.Linear(self.bb.num_features + 64, ncls)
    def forward(self, x, m): return self.head(torch.cat([self.bb(x), self.mm(m)], 1))

def run_fold_g(k, meta_t, imgs_, tf_tr, tf_ev, backbone="efficientnet_b0",
               epochs=12, bs=32, tta=False):
    tr_idx = df.index[df.fold != k]; te_idx = df.index[df.fold == k]
    tl = DataLoader(DSG(tr_idx, tf_tr, imgs_, meta_t), bs, shuffle=True, num_workers=2, drop_last=True)
    el = DataLoader(DSG(te_idx, tf_ev, imgs_, meta_t), 64, num_workers=2)
    model = FusionG(meta_t.shape[1], len(classes), backbone).to(dev)
    cnt = df.loc[tr_idx, "y"].value_counts().sort_index().values
    w = torch.tensor((cnt.sum() / (len(cnt) * cnt)) ** 0.5, dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w); scaler = torch.amp.GradScaler("cuda")
    for ep in range(epochs):
        if ep == 0:
            for p in model.bb.parameters(): p.requires_grad = False
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        if ep == 3:
            for p in model.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs - 3)
        model.train(); tot = 0
        for x, m, y in tl:
            x, m, y = x.to(dev), m.to(dev), y.to(dev); opt.zero_grad()
            with torch.autocast("cuda"): loss = lossf(model(x, m), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        if ep >= 3: sched.step()
        print(f"fold {k} ep {ep} loss {tot/len(tl):.3f}")
    model.eval(); P = []
    with torch.no_grad():
        for x, m, _ in el:
            x, m = x.to(dev), m.to(dev)
            views = (x, x.flip(3), x.flip(2)) if tta else (x,)
            P.append((sum(torch.softmax(model(v, m), 1) for v in views) / len(views)).cpu().numpy())
    return te_idx, np.concatenate(P), model

def run_all(name, **kw):
    o = np.zeros((len(df), len(classes)))
    for k in range(5):
        ti, pk, mk = run_fold_g(k, **kw)
        o[ti] = pk
        torch.save(mk.state_dict(), f"/kaggle/working/{name}_fold{k}.pt")
        np.save(f"/kaggle/working/oof_{name}.npy", o)
        print(f"=== {name} fold {k} done ===")
        del mk; torch.cuda.empty_cache()
    return o

def summary(o, name):
    y = df.y.values; pred = o.argmax(1)
    flag = [classes.index(c) for c in ["ACK", "BCC", "MEL", "SCC"]]
    is_flag = np.isin(y, flag); pr = o[:, flag].sum(1) >= 0.5      # threshold fixed in advance
    rec = recall_score(y, pred, average=None)
    print(f"{name}: acc {(pred==y).mean():.3f} | macroF1 {f1_score(y, pred, average='macro'):.3f} | "
          f"MEL rec {rec[classes.index('MEL')]:.3f} | SCC rec {rec[classes.index('SCC')]:.3f} | "
          f"sens {pr[is_flag].mean():.3f} spec {(~pr)[~is_flag].mean():.3f}")

In [11]:
for name in ["df", "imgs", "classes", "tf_train", "tf_eval", "norm", "dev"]:
    print(name, "OK" if name in globals() else "MISSING")
print("fold column:", "fold" in df.columns if "df" in globals() else "no df")

df OK
imgs OK
classes OK
tf_train OK
tf_eval OK
norm OK
dev OK
fold column: True


In [12]:
import os, glob
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

root = "/kaggle/input"
meta_path = glob.glob(f"{root}/**/metadata.csv", recursive=True)[0]
df = pd.read_csv(meta_path)

paths = {os.path.basename(p): p for p in glob.glob(f"{root}/**/*.png", recursive=True)}
df["path"] = df["img_id"].map(paths)
print("missing files:", df["path"].isna().sum())
df = df.dropna(subset=["path"]).reset_index(drop=True)
print(df["diagnostic"].value_counts())

missing files: 0
diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64


In [13]:
import torch, torch.nn as nn, timm, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

dev = "cuda"
classes = sorted(df.diagnostic.unique()); c2i = {c: i for i, c in enumerate(classes)}
df["y"] = df.diagnostic.map(c2i)
print(classes)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df["fold"] = -1
for k, (_, te) in enumerate(sgkf.split(df, df.y, groups=df.patient_id)):
    df.loc[te, "fold"] = k
print(pd.crosstab(df.diagnostic, df.fold))

imgs = [Image.open(p).convert("RGB").resize((256, 256)) for p in df.path]
print("cached", len(imgs))

['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']
fold          0    1    2    3    4
diagnostic                         
ACK         144  135  127  170  154
BCC         173  171  175  134  192
MEL          13    9    5   14   11
NEV          45   48   54   51   46
SCC          37   35   38   42   40
SEK          48   43   49   48   47
cached 2298


In [14]:
norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomVerticalFlip(), T.RandomRotation(20),
                      T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), norm])
tf_eval = T.Compose([T.CenterCrop(224), T.ToTensor(), norm])